# FFTW C2C Numba MPI

- https://numba.pydata.org/numba-doc/latest/user/withobjmode.html

In [ ]:
%%writefile test01.py
import numpy as np, time as tm
from numba import njit, objmode
from mpi4py_fft import PFFT, newDistArray
from mpi4py import MPI

N = 50
rs = np.array(0, dtype=np.complex128)
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
if rank == 0 : t0 = tm.time()    # <--- time measurement
if rank == 1 : print("1")

def uu() :
    return newDistArray(f, False)
def uh(u) :
    return f.forward(u, normalize=False)
@njit
def ff() :
    with objmode(u = 'complex128[:,:,:]') :  # annotate return type
        u = uu()
    if rank == 1 : print("3")
    for i in range(N) :
        for k in range(N) :
            for j in range(N) :
                u[i,j,k] = (i*N**2 + j*N  + k + 1)*1E-6
    if rank == 1 : print("4")
    with objmode(u_hat = 'complex128[:,:,:]') :  # annotate return type
        u_hat = uh(u)
        if rank == 1 : print("5")
    return np.array(np.sum(u_hat), dtype=np.complex128)

f = PFFT(comm, [N,N,N], dtype=np.complex128, backend='pyfftw')
s = ff()
if rank == 1 : print("2")
comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)
if rank == 0 : 
    t1 = tm.time()    # <--- time measurement
    print('S: {:.2f}    T: {:.4f} s'.format(rs, t1-t0))

In [2]:
%%writefile test02.py
import numpy as np, time as tm
from numba import njit, objmode
from mpi4py import MPI
from mpi4py_fft import PFFT, newDistArray

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
if rank == 0 : t0 = tm.time()    # <--- time measurement

N = 50
rs = np.array(0, dtype=np.complex128)
NA = np.array([N, N, N], dtype=int)
f = PFFT(comm, NA, dtype=np.complex128, backend='pyfftw')
u = newDistArray(f, False)
u[:] = np.random.random(u.shape).astype(u.dtype)
for i in range(u.shape[0]) :
    for k in range(u.shape[1]) :
        for j in range(u.shape[2]) :
            u[i,j,k] = (i*N**2 + j*N  + k + 1)*1E-6
u_hat = f.forward(u, normalize=False)

s = np.array(np.sum(u_hat), dtype=np.complex128)
comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)
if rank == 0 : 
    t1 = tm.time()    # <--- time measurement
    print('S: {:.2f}    T: {:.4f} s'.format(rs, t1-t0))

Overwriting teste02.py


In [3]:
! mpiexec -n 2 python teste02.py

S: 0.12+0.00j    T: 0.3476 s


In [30]:
%%writefile test03.py
import numpy as np, time as tm
from numba import njit, objmode
from mpi4py_fft import PFFT, newDistArray
from mpi4py import MPI

N = 500
rs = np.array(0, dtype=np.complex128)
comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()
if rank == 0 : t0 = tm.time()    # <--- time measurement

def uu() :
    return newDistArray(f, False)
@njit
def ff1() :
    with objmode(u = 'complex128[:,:,:]') :  # annotate return type
        u = uu()
    for i in range(u.shape[0]) :
        for j in range(u.shape[1]) :
            for k in range(u.shape[2]) :
                u[i,j,k] = (i*N**2 + j*N  + k + 1)*1E-6
    return u
@njit
def ff2(u_hat) :
    return np.array(np.sum(u_hat), dtype=np.complex128)

f = PFFT(comm, [N,N,N], dtype=np.complex128, backend='pyfftw')
u = ff1()
u_hat = f.forward(u, normalize=False)
s = ff2(u_hat)
comm.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)

if rank == 0 : 
    t1 = tm.time()    # <--- time measurement
    print('S={:.2f}  |  T={:.4f}  |  N={:0g}'.format(rs, t1-t0, size))

Overwriting bc2cs.py


In [31]:
! mpiexec -n 4 python bc2cs.py

S=125.00-0.00j  |  T=7.1371  |  N=4


In [22]:
%%writefile bc2cp.py
import numpy as np, time as tm
from numba import njit, objmode
from mpi4py_fft import PFFT, newDistArray
from mpi4py import MPI

comm = MPI.COMM_WORLD
size = comm.Get_size()
rank = comm.Get_rank()
if rank == 0 :
    t0 = tm.time()    # <--- time measurement

NN = 500
rs = np.array(0, dtype=np.complex128)

def uu() :
    return newDistArray(f, False)
def uf(u) :
    return f.forward(u, normalize=False)    
@njit
def ff() :
    with objmode(u = 'complex128[:,:,:]') :  # annotate return type
        u = uu()
    L, M, N = u.shape
    for i in range(L) :
        for j in range(M) :
            for k in range(N) :
                u[i, j, k] = (i*NN**2 + j*NN  + k + 1)*1E-6
    with objmode(u_hat = 'complex128[:,:,:]') :  # annotate return type
        u_hat = uf(u)
    return np.array(np.sum(u_hat), dtype=np.complex128)

# main
f = PFFT(comm, [NN,NN,NN], dtype=np.complex128, backend='pyfftw')
comm.Reduce([ff(), MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
            op=MPI.SUM, root=0)
if rank == 0 : 
    t1 = tm.time()    # <--- time measurement
    print('S={:.2f}  |  T={:.4f}  |  N={:0g}'.format(rs, t1-t0, size))

Overwriting bc2cp.py


In [2]:
! mpiexec -n 1 python bc2cp.py

S=125.00+0.00j  |  T=31.8943  |  N=1


In [4]:
! mpiexec -n 1 python bc2cp.py

S=125.00+0.00j  |  T=23.5482  |  N=1


In [23]:
! mpiexec -n 1 python bc2cp.py

S=125.00+0.00j  |  T=35.9464  |  N=1


In [39]:
! mpiexec -n 16 python bc2cp.py

S=125.00-0.00j  |  T=3.1886  |  N=16


In [1]:
%%writefile bc2cp.py
import numpy as np, time as tm
from numba import njit, objmode
from mpi4py_fft import PFFT, newDistArray
from mpi4py import MPI

def uu() : 
    return newDistArray(f, False)
def uf(u) :
    return f.forward(u, normalize=False)    
@njit
def ff() :
    rs = np.array(0, dtype=np.complex128)
    with objmode(size = 'intc', rank = 'intc', t0 = 'double') :
        size = MPI.COMM_WORLD.Get_size()
        rank = MPI.COMM_WORLD.Get_rank()
        t0 = tm.time()
    with objmode(u = 'complex128[:,:,:]') :  # annotate return type
        u = uu()
    L, M, N = u.shape
    for i in range(L) :
        for j in range(M) :
            for k in range(N) :
                u[i, j, k] = (i*NN**2 + j*NN  + k + 1)*1E-6
    with objmode(u_hat = 'complex128[:,:,:]') :  # annotate return type
        u_hat = uf(u)
    s = np.array(np.sum(u_hat), dtype=np.complex128)
    with objmode() :
        MPI.COMM_WORLD.Reduce([s, MPI.DOUBLE_COMPLEX], [rs, MPI.DOUBLE_COMPLEX],
                op=MPI.SUM, root=0)
    if not rank :
        with objmode() :
            t1 = tm.time()    # <--- time measurement
            print('S={:.2f}  |  T={:.4f}  |  N={:0g}'.format(rs, t1-t0, size))

NN = 500
# PFFT should be outside the numba function due to the "class" return
f = PFFT(MPI.COMM_WORLD, [NN,NN,NN], dtype=np.complex128, backend='pyfftw')
ff()

Overwriting bc2cp.py


In [142]:
! mpiexec -n 1 python bc2cp.py

S=125.00+0.00j  |  T=17.0084  |  N=1


In [141]:
! mpiexec -n 16 python bc2cp.py

S=125.00-0.00j  |  T=1.8064  |  N=16


In [1]:
! mpiexec -n 1 python bc2cp.py

S=125.00+0.00j  |  T=18.1854  |  N=1


In [2]:
! mpiexec -n 16 python bc2cp.py

S=125.00-0.00j  |  T=2.1173  |  N=16


In [2]:
! cp bc2cp.py /scratch${PWD#"/prj"}

In [22]:
! srun --mpi=list

srun: MPI types are...
srun: mpi/mpich1_p4
srun: mpi/mvapich
srun: mpi/none
srun: mpi/openmpi
srun: mpi/pmi2
srun: mpi/mpichgm
srun: mpi/lam
srun: mpi/mpich1_shmem
srun: mpi/mpichmx


do not work: pmi2, openmpi, mpich1_p4, none, mpichmx

In [93]:
%%writefile bc2cp.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name bc2cp       # Job name
#SBATCH --time=00:02:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes
#   NTASKS: 1, 4, 9, 16, 36, 49, 64, 81
echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Environment
cd
cd /scratch${PWD#"/prj"}/
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
source  /scratch/app/anaconda3/2020.11/etc/profile.d/conda.sh
conda activate /scratch/.../.../anaconda3/envs/
conda activate --stack ./envs
cd  fft
                                              
# Executable
EXEC="python bc2cp.py"

# Start
echo '$ srun  --mpi=pmi2  -n' $SLURM_NTASKS  ${EXEC##*/}
echo '-- output -----------------------------'
srun  --mpi=pmi2  -n $SLURM_NTASKS  $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting bc2cp.srm


In [89]:
! sbatch --partition=cpu_dev --ntasks=16 bc2cp.srm

Submitted batch job 876344


In [91]:
! squeue -n bc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [92]:
! cat /scratch${PWD#"/prj"}/slurm-876344.out

- Job ID: 876344
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1271
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 16 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=1.6883  |  N=16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 1 of (1, 4, 9, 16, 36, 49, 64, 81)

In [95]:
! sbatch --ntasks=1 bc2cp.srm
! sbatch --ntasks=1 bc2cp.srm
! sbatch --ntasks=1 bc2cp.srm

Submitted batch job 876345
Submitted batch job 876346
Submitted batch job 876347


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-876345.out
! cat /scratch${PWD#"/prj"}/slurm-876346.out
! cat /scratch${PWD#"/prj"}/slurm-876347.out

- Job ID: 876345
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 1 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=12.6436  |  N=1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876346
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 1 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=12.6681  |  N=1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876347
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
Not a conda environment: /scratch/amp

### 4 of (1, 4, 9, 16, 36, 49, 64, 81)

In [96]:
! sbatch --ntasks=4 bc2cp.srm
! sbatch --ntasks=4 bc2cp.srm
! sbatch --ntasks=4 bc2cp.srm

Submitted batch job 876348
Submitted batch job 876349
Submitted batch job 876350


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-876348.out
! cat /scratch${PWD#"/prj"}/slurm-876349.out
! cat /scratch${PWD#"/prj"}/slurm-876350.out

- Job ID: 876348
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 4 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=4.7404  |  N=4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876349
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 4 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=4.7180  |  N=4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876350
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampem

### 9 of (1, 4, 9, 16, 36, 49, 64, 81)

In [97]:
! sbatch --ntasks=9 bc2cp.srm
! sbatch --ntasks=9 bc2cp.srm
! sbatch --ntasks=9 bc2cp.srm

Submitted batch job 876351
Submitted batch job 876352
Submitted batch job 876353


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-876351.out
! cat /scratch${PWD#"/prj"}/slurm-876352.out
! cat /scratch${PWD#"/prj"}/slurm-876353.out

- Job ID: 876351
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 9 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=2.4057  |  N=9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876352
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 9 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=2.4208  |  N=9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876353
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampem

### 16 of (1, 4, 9, 16, 36, 49, 64, 81)

In [98]:
! sbatch --ntasks=16 bc2cp.srm
! sbatch --ntasks=16 bc2cp.srm
! sbatch --ntasks=16 bc2cp.srm

Submitted batch job 876354
Submitted batch job 876355
Submitted batch job 876356


In [5]:
! cat /scratch${PWD#"/prj"}/slurm-876354.out
! cat /scratch${PWD#"/prj"}/slurm-876355.out
! cat /scratch${PWD#"/prj"}/slurm-876356.out

- Job ID: 876354
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 16 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=1.6501  |  N=16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876355
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 16 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=1.6454  |  N=16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876356
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1325
Not a conda environment: /scratc

### 36 of (1, 4, 9, 16, 36, 49, 64, 81)

In [99]:
! sbatch --ntasks=36 bc2cp.srm
! sbatch --ntasks=36 bc2cp.srm
! sbatch --ntasks=36 bc2cp.srm

Submitted batch job 876357
Submitted batch job 876358
Submitted batch job 876359


In [6]:
! cat /scratch${PWD#"/prj"}/slurm-876357.out
! cat /scratch${PWD#"/prj"}/slurm-876358.out
! cat /scratch${PWD#"/prj"}/slurm-876359.out

- Job ID: 876357
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1312 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 36 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=5.3048  |  N=36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876358
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1312 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 36 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=5.4797  |  N=36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876359
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1312 sdumont1

### 49 of (1, 4, 9, 16, 36, 49, 64, 81)

In [100]:
! sbatch --ntasks=49 bc2cp.srm
! sbatch --ntasks=49 bc2cp.srm
! sbatch --ntasks=49 bc2cp.srm

Submitted batch job 876360
Submitted batch job 876361
Submitted batch job 876362


In [7]:
! cat /scratch${PWD#"/prj"}/slurm-876360.out
! cat /scratch${PWD#"/prj"}/slurm-876361.out
! cat /scratch${PWD#"/prj"}/slurm-876362.out

- Job ID: 876360
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303 sdumont1309 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 49 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=6.7562  |  N=49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876361
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303 sdumont1309 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 49 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=7.1367  |  N=49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876362
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the j

### 64 of (1, 4, 9, 16, 36, 49, 64, 81)

In [101]:
! sbatch --ntasks=64 bc2cp.srm
! sbatch --ntasks=64 bc2cp.srm
! sbatch --ntasks=64 bc2cp.srm

Submitted batch job 876363
Submitted batch job 876364
Submitted batch job 876365


In [8]:
! cat /scratch${PWD#"/prj"}/slurm-876363.out
! cat /scratch${PWD#"/prj"}/slurm-876364.out
! cat /scratch${PWD#"/prj"}/slurm-876365.out

- Job ID: 876363
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303 sdumont1309 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 64 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=6.0261  |  N=64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876364
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303 sdumont1309 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 64 python bc2cp.py
-- output -----------------------------
S=125.00-0.00j  |  T=6.7010  |  N=64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876365
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the j

### 81 of (1, 4, 9, 16, 36, 49, 64, 81)

In [102]:
! sbatch --ntasks=81 bc2cp.srm
! sbatch --ntasks=81 bc2cp.srm
! sbatch --ntasks=81 bc2cp.srm

Submitted batch job 876366
Submitted batch job 876367
Submitted batch job 876368


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-876366.out
! cat /scratch${PWD#"/prj"}/slurm-876367.out
! cat /scratch${PWD#"/prj"}/slurm-876368.out

- Job ID: 876366
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303 sdumont1309 sdumont1312 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 81 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=6.2174  |  N=81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876367
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1303 sdumont1309 sdumont1312 sdumont1325
Not a conda environment: /scratch/ampemi/pedro.santos2/anaconda3/envs
$ srun  --mpi=pmi2  -n 81 python bc2cp.py
-- output -----------------------------
S=125.00+0.00j  |  T=6.4407  |  N=81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 876368
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of 

In [1]:
! squeue -n bc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            876350  cpu_small  PD  0:00     1    4
            876349  cpu_small  PD  0:00     1    4
            876348  cpu_small  PD  0:00     1    4
            876347  cpu_small  PD  0:00     1    1
            876346  cpu_small  PD  0:00     1    1
            876345  cpu_small  PD  0:00     1    1
            876368  cpu_small  PD  0:00     4   81
            876367  cpu_small  PD  0:00     4   81
            876366  cpu_small  PD  0:00     4   81
            876365  cpu_small  PD  0:00     3   64
            876364  cpu_small  PD  0:00     3   64
            876363  cpu_small  PD  0:00     3   64
            876362  cpu_small  PD  0:00     3   49
            876361  cpu_small  PD  0:00     3   49
            876360  cpu_small  PD  0:00     3   49
            876359  cpu_small  PD  0:00     2   36
            876358  cpu_small  PD  0:00     2   36
            876357  cpu_small  PD  0:00     2   36
            876356  cpu_small  

In [9]:
! squeue -n bc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            876366  cpu_small  PD  0:00     4   81
            876367  cpu_small  PD  0:00     4   81
            876368  cpu_small  PD  0:00     4   81


In [5]:
! scancel  -n bc2cp

In [1]:
! squeue -n bc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
